<!-- Cache bust 14_concurrency_and_parallelism_notebook -->

# Concurrency & Parallelism

***

### 🔹 1. CPU-Bound vs I/O-Bound Tasks
To write fast and responsive programs, we must understand the bottleneck of our execution:

* **I/O-Bound Tasks:** The program spends most of its time waiting for external responses (e.g., fetching web pages, reading/writing disk, database queries).
* **CPU-Bound Tasks:** The program spends most of its time doing math and logical operations (e.g., model training, matrix multiplication, image processing, parsing).

Python provides three distinct approaches for concurrency: **Multithreading**, **Multiprocessing**, and **Asynchronous Programming (asyncio)**. Which one to choose depends entirely on the type of bottleneck.

***

### 🔹 2. The Global Interpreter Lock (GIL)
The **GIL (Global Interpreter Lock)** is a mutex that protects access to Python objects, preventing multiple native threads from executing Python bytecodes at once.
* **Why it exists:** Simplifies memory management (especially reference counting) and makes C-extensions easier to write.
* **Implication:** In Python, standard **Multithreading cannot run code in parallel across multiple CPU cores**. If you have CPU-bound tasks, multithreading might actually make your code slower due to context switching overhead.
* **How to bypass GIL:**
  1. Use **Multiprocessing** (launches separate OS processes, each with its own GIL).
  2. Offload heavy computation to compiled C-extensions (like NumPy, PyTorch, Scikit-learn, which release the GIL internally during heavy operations).

***

### 🔹 3. Multithreading (threading module)
Multithreading is excellent for I/O-bound tasks where threads spend most of their time waiting.

#### ⚠️ The Danger: Race Conditions
Since threads share the same memory space, if two threads try to modify a shared variable at the same time, the data can get corrupted. We use **Locks** (`threading.Lock`) to prevent this by ensuring only one thread can access a block of code at a time.

In [ ]:
import threading
import time

# Shared resource
counter = 0
lock = threading.Lock()

def increase():
    global counter
    for _ in range(100000):
        # Without lock, counter would be corrupted
        with lock:
            counter += 1

# Creating threads
t1 = threading.Thread(target=increase)
t2 = threading.Thread(target=increase)

# Start threads
t1.start()
t2.start()

# Wait for threads to finish
t1.join()
t2.join()

print("Final Counter (Should be 200000):", counter)

***

### 🔹 4. Multiprocessing (multiprocessing module)
Multiprocessing bypasses the GIL completely by spawning independent OS processes, each having its own memory space and Python interpreter.

* Excellent for **CPU-bound tasks**.
* Use a **Process Pool** (`multiprocessing.Pool`) to easily distribute work across multiple CPU cores.

In [ ]:
import multiprocessing
import time

def compute_square(n):
    # Simulating a CPU intensive computation
    return n * n

if __name__ == "__main__": # Mandatory for multiprocessing on Windows!
    numbers = [1, 2, 3, 4, 5, 6, 7, 8]
    
    # Measure execution with a process pool
    # Pool(processes=4) creates 4 worker processes
    start = time.perf_counter()
    with multiprocessing.Pool(processes=4) as pool:
        results = pool.map(compute_square, numbers)
    end = time.perf_counter()
    
    print("Results:", results)
    print(f"Time taken in parallel: {end - start:.5f} seconds")

***

### 🔹 5. Asynchronous Programming (asyncio)
**`asyncio`** is a framework that writes concurrent code using the `async` / `await` syntax.
* Unlike multithreading, it uses **cooperative multitasking** within a single thread.
* An **Event Loop** manages and schedules tasks. Tasks cooperate by yielding control back to the loop when they are waiting for I/O operations.
* **Coroutines:** Functions defined with `async def`. They can pause execution using `await`.

#### 🧠 Why use asyncio over threads?
* Extremely lightweight: Can run tens of thousands of concurrent tasks (threads are limited to a few hundred due to OS memory limits).
* Avoids race conditions (no locks needed) since execution happens in a single thread.

In [ ]:
import asyncio

async def fetch_api_data(endpoint, delay):
    print(f"[API] Fetching from {endpoint}...")
    # asyncio.sleep simulates waiting for a network response (non-blocking)
    await asyncio.sleep(delay)
    print(f"[API] Received data from {endpoint}!")
    return {endpoint: "success"}

async def main():
    # Schedule multiple coroutines to run concurrently
    task1 = fetch_api_data("users_endpoint", 2)
    task2 = fetch_api_data("products_endpoint", 1)
    task3 = fetch_api_data("analytics_endpoint", 1.5)
    
    # Wait for all tasks to complete
    results = await asyncio.gather(task1, task2, task3)
    print("All API Results:", results)

# To run a coroutine in a script, use asyncio.run():
# asyncio.run(main()) # Note: In Jupyter, the event loop is already running, 
# so we run it using: await main()
await main()

***

### 🔹 6. How to Choose Concurrency Models?

| Model | Bottleneck Type | Memory Sharing | Overhead | GIL Limitation |
| :--- | :--- | :--- | :--- | :--- |
| **Multithreading** | I/O-Bound | Yes (Shared memory) | Low | Restricted by GIL |
| **Multiprocessing** | CPU-Bound | No (Separate memory) | High | Bypasses GIL |
| **Asyncio** | I/O-Bound (High Scale) | Yes (Single-thread) | Extremely Low | Restricted by GIL |

***

### 📝 Practice Questions

1. **Parallel Downloader Simulation:** Write a script using the `threading` module to simulate downloading 5 files concurrently. Each download should take a different number of seconds (simulated using `time.sleep()`).
2. **Parallel Prime Search:** Write a multiprocessing script that searches for prime numbers inside multiple ranges of integers concurrently (e.g. searching range 1-100000, 100001-200000).
3. **Async Batch Web Scraper:** Simulate an asynchronous scraper using `asyncio` that fetches HTML from a list of 10 mock URLs. It should fetch them in batches of 3 concurrently (Hint: use `asyncio.Semaphore(3)` or split list into batches).